# Core ML and data handling libraries

In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np
from collections import Counter
import warnings
import os
import json
import re

# Image Processing
import cv2
from skimage.feature import hog

# Graph Analysis & Visualization
import networkx as nx
import plotly.graph_objects as go

# Google Colab
from google.colab import drive
import matplotlib.pyplot as plt

# Suppress minor warnings for a cleaner output
warnings.filterwarnings("ignore", category=UserWarning)


#  DATA LOADING CLASS     

# EnhancedMultimodalDataLoader

## 📖 Overview
A comprehensive data loading utility class designed to handle **multimodal datasets** containing both **image data** and **structured JSON knowledge bases**.  
This class is specifically designed for **machine learning workflows** that require unified access to **visual and textual data sources**.

---

## 🎯 Purpose

- **Unified Data Access**: Combines image loading with JSON-based knowledge retrieval  
- **Dataset Validation**: Automatically verifies data paths and reports missing components  
- **Flexible Structure**: Supports hierarchical image organization with automatic label extraction  
- **Error Handling**: Robust processing with detailed logging and graceful error recovery  

---

## ✨ Key Features

### 🖼️ Image Processing
- Recursively loads images from nested subdirectories  
- Automatic format detection (`.png`, `.jpg`, `.jpeg`)  
- Standardized preprocessing:  
  - RGB conversion  
  - Resizing to **224×224**  
- Label extraction from directory structure  
- Batch processing with progress reporting  

### 📄 JSON Knowledge Integration
- Loads **question–answer pairs** for training/evaluation  
- Incorporates **structured knowledge bases**  
- UTF-8 encoding support for international datasets  
- Flexible JSON schema handling  

### 🔧 Data Unification
- Creates structured datasets ready for ML pipelines  
- Maintains data relationships between images and metadata  
- Returns standardized **dictionary format** for easy consumption  

---

## 💡 Usage Scenarios
- **Medical AI**: Disease diagnosis systems with image + knowledge integration  
- **Educational Tools**: Visual learning platforms with Q&A components  
- **Research Projects**: Academic datasets requiring multimodal preprocessing  
- **Production Systems**: Scalable data loading for inference pipelines  

---

## 🏗️ Architecture Benefits
- **Modular Design**: Separate methods for different data types  
- **Path Flexibility**: Configurable base directories for different environments  
- **Validation First**: Early path verification prevents runtime errors  
- **Memory Efficient**: Loads data on-demand with proper resource management  

---




In [ ]:
class EnhancedMultimodalDataLoader:
    """Handles finding, loading, and unifying the raw image and JSON data."""
    def __init__(self, base_path='/content/drive/MyDrive/GNN_dataset/'):
        self.base_path = base_path
        self.images_path = os.path.join(base_path, 'images')
        self.test_path = os.path.join(base_path, 'test') # Path for JSON files
        self.image_data = []
        self.qa_pairs = []
        self.knowledge_base = []
        self.verify_paths()

    def verify_paths(self):
        """Verifies the dataset paths exist."""
        print("\nVerifying dataset paths...")
        if not os.path.exists(self.images_path):
            print(f"Error: The 'images' directory was not found at {self.images_path}.")
        else:
            print(f"Images path found: {self.images_path}")
        if not os.path.exists(self.test_path):
            print(f" Warning: The 'test' directory (for JSONs) was not found at {self.test_path}.")
        else:
            print(f"JSON path found: {self.test_path}")

    def load_images(self):
        """Loads images by walking through subdirectories."""
        print("\nLoading images from subfolders...")
        if not os.path.exists(self.images_path): return

        for root, _, files in os.walk(self.images_path):
            label = os.path.basename(root)
            if label == 'images': continue

            image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            if not image_files: continue
            print(f"  - Processing folder '{label}': Found {len(image_files)} images.")

            for img_file in image_files:
                try:
                    img_path = os.path.join(root, img_file)
                    image = cv2.imread(img_path)
                    if image is not None:
                        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                        image = cv2.resize(image, (224, 224))
                        self.image_data.append({'image': image, 'label': label})
                except Exception as e:
                    print(f"    - Skipping file {img_file} due to error: {e}")

        print(f"\n Successfully loaded {len(self.image_data)} images from all subfolders.")

    def load_json_data(self):
        """Loads supplementary JSON data from the 'test' directory."""
        print("\nLoading JSON knowledge base...")
        json_files = {
            'qa_pairs': os.path.join(self.test_path, 'disease_diagnosis.json'),
            'knowledge': os.path.join(self.test_path, 'disease_knowledge.json')
        }
        for key, path in json_files.items():
            if os.path.exists(path):
                try:
                    with open(path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        if key == 'qa_pairs': self.qa_pairs = data
                        if key == 'knowledge': self.knowledge_base = data
                        print(f"  - Loaded {len(data)} entries from {os.path.basename(path)}")
                except Exception as e:
                    print(f"  - Error loading {path}: {e}")
            else:
                print(f"  - Warning: JSON file not found at {path}")

    def create_unified_dataset(self):
        """Loads all data and returns it in a structured dictionary."""
        self.load_images()
        self.load_json_data()
        if not self.image_data: return None
        return {
            'images': [item['image'] for item in self.image_data],
            'labels': [item['label'] for item in self.image_data],
            'qa_pairs': self.qa_pairs,
            'knowledge': self.knowledge_base
        }

# FEATURE EXTRACTION CLASS

# 📊 Data Processing Pipeline Classes

---

## 1️⃣ EnhancedMultimodalDataLoader

### 📖 Overview
A comprehensive data loading utility class designed to handle **multimodal datasets** containing both **image data** and **structured JSON knowledge bases**.  
This class is specifically designed for **machine learning workflows** that require unified access to **visual and textual data sources**.

---

### 🎯 Purpose
- **Unified Data Access**: Combines image loading with JSON-based knowledge retrieval  
- **Dataset Validation**: Automatically verifies data paths and reports missing components  
- **Flexible Structure**: Supports hierarchical image organization with automatic label extraction  
- **Error Handling**: Robust processing with detailed logging and graceful error recovery  

---

### ✨ Key Features

#### 🖼️ Image Processing
- Recursively loads images from nested subdirectories  
- Automatic format detection (`.png`, `.jpg`, `.jpeg`)  
- Standardized preprocessing:  
  - RGB conversion  
  - Resizing to **224×224**  
- Label extraction from directory structure  
- Batch processing with progress reporting  

#### 📄 JSON Knowledge Integration
- Loads **question–answer pairs** for training/evaluation  
- Incorporates **structured knowledge bases**  
- UTF-8 encoding support for international datasets  
- Flexible JSON schema handling  

#### 🔧 Data Unification
- Creates structured datasets ready for ML pipelines  
- Maintains data relationships between images and metadata  
- Returns standardized **dictionary format** for easy consumption  

---



In [ ]:
class FeatureExtractor:
    """Converts raw images and text into numerical feature vectors."""
    def __init__(self, dataset):
        self.dataset = dataset
        self.tfidf_vectorizer = TfidfVectorizer(max_features=200, stop_words='english')
        print("\nFeatureExtractor initialized.")

    def extract_image_features(self):
        """Extracts HOG features from images."""
        print("Extracting image features (HOG)...")
        hog_features = [hog(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), orientations=8, pixels_per_cell=(16, 16),
                            cells_per_block=(1, 1), visualize=False) for img in self.dataset['images']]
        return np.array(hog_features)

    def extract_text_features(self):
        """Creates context-aware TF-IDF features for each image using the JSON knowledge base."""
        print("Extracting context-aware text features from JSON...")
        corpus = [json.dumps(item).lower() for item in self.dataset.get('qa_pairs', []) + self.dataset.get('knowledge', [])]
        if not corpus:
            print("Warning: No JSON data found. Returning zero vectors for text features.")
            return np.zeros((len(self.dataset['labels']), 1))

        self.tfidf_vectorizer.fit(corpus)
        all_text_features = []
        for label in self.dataset['labels']:
            keywords = set(re.split(r'[_ ]', label.lower()))
            profile_docs = [doc for doc in corpus if any(key in doc for key in keywords)]
            if profile_docs:
                profile_vector = self.tfidf_vectorizer.transform(profile_docs)
                final_vector = profile_vector.mean(axis=0)
            else:
                final_vector = np.zeros((1, len(self.tfidf_vectorizer.get_feature_names_out())))
            all_text_features.append(np.asarray(final_vector).flatten())
        return np.array(all_text_features)

    def create_all_features(self):
        """Runs all feature extraction and returns a dictionary of numerical arrays."""
        image_features = self.extract_image_features()
        text_features = self.extract_text_features()
        labels = self.dataset['labels']
        combined_features = np.hstack([image_features, text_features])

        print(f"Feature extraction complete. Shapes: Image={image_features.shape}, Text={text_features.shape}")
        return {'image_features': image_features, 'text_features': text_features, 'combined_features': combined_features, 'labels': labels}


# 🔗 Graph Analysis and Training Classes

---

## 3️⃣ GraphBuilder

### 📖 Overview
A sophisticated **graph construction and visualization engine** that transforms **feature vectors into network representations**.  
This class creates meaningful **connections between data points** based on similarity metrics and provides **interactive visualizations** for network analysis.

---

### 🎯 Purpose
- **Network Construction**: Builds similarity-based graphs from numerical features  
- **Interactive Visualization**: Creates color-coded, interactive network plots  
- **Relationship Discovery**: Reveals hidden patterns and clusters in multimodal data  
- **Scalable Analysis**: Handles large datasets with intelligent subgraph extraction  

---

### ✨ Key Features

#### 📊 Similarity Graph Construction
- **Cosine Similarity**: Measures feature vector similarity between all data points  
- **Threshold-Based Connections**: Configurable similarity threshold (default: `0.7`)  
- **Weighted Edges**: Edge weights represent similarity strength  
- **Efficient Processing**: Optimized pairwise similarity computation  

#### 🎨 Interactive Visualization
- **Color-Coded Categories**: Automatic color assignment based on merged labels  
- **Hover Information**: Detailed node information on mouse hover  
- **Spring Layout**: Aesthetically pleasing force-directed positioning  
- **Subgraph Intelligence**: Automatically focuses on largest connected components  
- **Performance Optimization**: Limits visualization to manageable node counts (`max 75`)  

---

## 🚀 Summary
The **GraphBuilder** class bridges the gap between **numerical feature vectors** and **graph-based insights**.  
It enables **network-driven analysis**, **cluster discovery**, and **intuitive visualization** for multimodal datasets, making it a cornerstone for **GNN-based workflows**.

## 4️⃣ GraphAnalyzer

### 📖 Overview
A **feature extraction utility** that transforms **graph structures into numerical representations** suitable for machine learning algorithms.  
This class bridges **graph theory with traditional ML** by extracting meaningful **structural properties** from network data.

---

### 🎯 Purpose
- **Structural Feature Extraction**: Converts graph topology into numerical vectors  
- **Centrality Analysis**: Measures node importance and connectivity patterns  
- **Graph-ML Integration**: Provides graph features for enhanced ML models  
- **Network Insights**: Quantifies structural properties for deeper analysis  

---

### ✨ Key Features

#### 📈 Node-Level Features
- **Degree**: Number of direct connections per node  
- **Degree Centrality**: Normalized measure of node connectivity  
- **Clustering Coefficient**: Local network density around each node  
- **Standardized Output**: Consistent **3-feature vectors per node**  

---

## 🚀 Summary
The **GraphAnalyzer** class enables the **translation of complex graph structures into ML-ready feature sets**.  
By combining **node-level insights** with **graph-level properties**, it strengthens downstream tasks such as **classification, clustering, and predictive modeling**.


In [ ]:
class GraphBuilder:
    """Builds and visualizes graphs from dataset features."""
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
        self.graphs = {}
        print("\nGraphBuilder initialized.")

    def build_similarity_graph(self, threshold=0.7):
        """Builds a graph connecting nodes with high feature similarity."""
        print("Building similarity graph...")
        G = nx.Graph()
        for i, label in enumerate(self.labels):
            G.add_node(i, label=label)

        similarity_matrix = cosine_similarity(self.features)
        for i in range(len(self.features)):
            for j in range(i + 1, len(self.features)):
                if similarity_matrix[i, j] > threshold:
                    G.add_edge(i, j, weight=similarity_matrix[i, j])
        self.graphs['similarity'] = G
        print(f"Similarity graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")

    def visualize_graph(self, graph_name, max_nodes=75):
        """Creates an interactive, color-coded visualization of the specified graph."""
        print(f"\nVisualizing '{graph_name}' graph...")
        if graph_name not in self.graphs or self.graphs[graph_name].number_of_nodes() == 0:
            print(f"Graph '{graph_name}' is empty or not found. Skipping visualization.")
            return

        G = self.graphs[graph_name]

        if G.number_of_nodes() > max_nodes:
            components = list(nx.connected_components(G))
            if components:
                largest_cc = max(components, key=len)
                nodes_to_show = list(largest_cc)[:max_nodes]
                G = G.subgraph(nodes_to_show)
                print(f"Graph is large. Visualizing a subgraph of {len(nodes_to_show)} nodes.")
            else:
                print("Graph has no connected components to visualize.")
                return

        pos = nx.spring_layout(G, k=0.8, iterations=50)

        edge_x, edge_y = [], []
        for edge in G.edges():
            x0, y0 = pos[edge[0]]
            x1, y1 = pos[edge[1]]
            edge_x.extend([x0, x1, None])
            edge_y.extend([y0, y1, None])

        node_x, node_y, hover_text, node_color = [], [], [], []

        original_labels = nx.get_node_attributes(G, 'label')
        merged_labels_map = {node: label.split('_')[0] for node, label in original_labels.items()}
        unique_merged_labels = sorted(list(set(merged_labels_map.values())))

        # CORRECTED: Use updated Matplotlib API and format color strings correctly
        cmap = plt.colormaps.get_cmap('viridis')
        colors = cmap(np.linspace(0, 1, len(unique_merged_labels)))
        color_map = {}
        for i, label in enumerate(unique_merged_labels):
            r, g, b, a = (colors[i] * 255).astype(int)
            color_map[label] = f'rgba({r},{g},{b},{a})'

        for node in G.nodes():
            x, y = pos[node]
            node_x.append(x)
            node_y.append(y)

            orig_label = original_labels.get(node, 'N/A')
            merged_label = merged_labels_map.get(node, 'N/A')

            hover_text.append(f'<b>{orig_label}</b><br>Group: {merged_label}<br>Node ID: {node}')
            node_color.append(color_map.get(merged_label, 'rgba(128,128,128,0.8)'))

        edge_trace = go.Scatter(x=edge_x, y=edge_y, line=dict(width=0.5, color='#888'), hoverinfo='none', mode='lines')
        node_trace = go.Scatter(x=node_x, y=node_y, mode='markers', hoverinfo='text', text=hover_text,
                                marker=dict(showscale=False, color=node_color, size=10, line=dict(width=2, color='black')))

        fig = go.Figure(data=[edge_trace, node_trace],
                        layout=go.Layout(title=f'<b>{graph_name.replace("_", " ").title()} Network (Colored by Category)</b>', titlefont_size=16,
                                         showlegend=False, hovermode='closest',
                                         margin=dict(b=20, l=5, r=5, t=40),
                                         xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                                         yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))
        fig.show()

class GraphAnalyzer:
    """Extracts features from graphs."""
    def __init__(self, graphs):
        self.graphs = graphs
        print("GraphAnalyzer initialized.")

    def extract_graph_features_for_all_nodes(self, graph_name):
        """Extracts structural features for all nodes in a graph."""
        if graph_name not in self.graphs: return None
        G = self.graphs[graph_name]
        print(f"Extracting features from '{graph_name}' graph...")
        degree_centrality = nx.degree_centrality(G)
        all_features = [[G.degree(node_id), degree_centrality.get(node_id, 0), nx.clustering(G, node_id)]
                        for node_id in sorted(G.nodes())]
        return np.array(all_features)

## 5️⃣ RobustMultimodalTrainer

### 📖 Overview
A comprehensive **machine learning evaluation framework** designed for **multimodal datasets with graph-enhanced features**.  
This class provides **robust model comparison** using cross-validation and handles the complete **ML pipeline** from **data preparation to performance reporting**.

---

### 🎯 Purpose
- **Multimodal Integration**: Combines image, text, and graph features seamlessly  
- **Robust Evaluation**: Uses **Leave-One-Out Cross-Validation (LOOCV)** for reliable estimates  
- **Feature Enhancement**: Augments traditional features with graph-based insights  
- **Performance Analysis**: Comprehensive model comparison and reporting  

---

### ✨ Key Features

#### 🔄 Data Preparation Pipeline
- **Label Encoding**: Converts string labels to numerical format  
- **Feature Scaling**: `StandardScaler` normalization for all feature types  
- **Class Merging**: Intelligent label simplification (e.g., *Apple_Scab* → *Apple*)  
- **Feature Stacking**: Horizontal concatenation of multimodal features  

#### 🧠 Enhanced Feature Sets
- **Image Features**: HOG-based visual representations  
- **Text Features**: TF-IDF textual embeddings  
- **Combined Features**: Fusion of image and text vectors  
- **Graph-Enhanced**: Combined features augmented with structural properties  
- **Automatic Scaling**: Individual scalers for each feature type  

#### 🎯 Model Evaluation Framework
- **Leave-One-Out Cross-Validation (LOOCV):** Most robust validation for small datasets  
- **Multiple Algorithms:** Random Forest and SVM with RBF kernel  
- **Balanced Training:** Class weight adjustment for imbalanced datasets  
- **Parallel Processing:** Multi-core evaluation for efficiency  

#### 📊 Performance Analysis
- **Comprehensive Reporting:** Tabular results with statistical summaries  
- **Best Model Selection:** Automatic identification of top-performing configuration  
- **Class Distribution Analysis:** Before/after merging statistics  
- **Reproducible Results:** Fixed random seeds for consistent evaluation  

---

## 🚀 Summary
The **RobustMultimodalTrainer** provides a **complete ML experimentation framework** for multimodal + graph-enhanced data.  
By combining **rigorous validation**, **advanced feature fusion**, and **detailed reporting**, it ensures reliable insights for both **research** and **production-ready AI systems**.

In [ ]:
class RobustMultimodalTrainer:
    """A trainer for evaluating multimodal features with robust validation."""
    def __init__(self, features_dict, graph_analyzer=None):
        self.image_features = features_dict['image_features']
        self.text_features = features_dict['text_features']
        self.combined_features = features_dict['combined_features']
        self.labels = np.array(features_dict['labels'])
        self.graph_analyzer = graph_analyzer
        self.results = {}
        self.feature_sets = {}
        self.scalers = {name: StandardScaler() for name in ['Image', 'Text', 'Combined', 'Graph', 'Enhanced']}
        print("\nRobustMultimodalTrainer initialized.")

    def merge_classes_by_prefix(self, separator='_'):
        """Merges labels based on a prefix (e.g., 'Apple_Scab' -> 'Apple')."""
        print("\nMerging classes to simplify the classification task...")
        self.analyze_class_distribution(stage="Before Merging")
        self.labels = np.array([label.split(separator)[0] for label in self.labels])
        self.analyze_class_distribution(stage="After Merging")

    def prepare_data(self):
        """Scales all feature sets and prepares them for evaluation."""
        print("\nPreparing and scaling all feature sets...")
        self.label_encoder = LabelEncoder()
        self.encoded_labels = self.label_encoder.fit_transform(self.labels)

        self.feature_sets['Image'] = self.scalers['Image'].fit_transform(self.image_features)
        self.feature_sets['Text'] = self.scalers['Text'].fit_transform(self.text_features)
        scaled_combined = self.scalers['Combined'].fit_transform(self.combined_features)
        self.feature_sets['Combined'] = scaled_combined

        if self.graph_analyzer:
            graph_features = self.graph_analyzer.extract_graph_features_for_all_nodes('similarity')
            if graph_features is not None and graph_features.shape[0] == scaled_combined.shape[0]:
                scaled_graph = self.scalers['Graph'].fit_transform(graph_features)
                enhanced_features = np.hstack([scaled_combined, scaled_graph])
                self.feature_sets['Graph-Enhanced'] = self.scalers['Enhanced'].fit_transform(enhanced_features)
                print(f"Graph-Enhanced features created, shape: {enhanced_features.shape}")

        print("Feature preparation complete.")

    def evaluate_all_models(self):
        """Evaluates models on all feature sets using LOOCV."""
        print("\nEvaluating models with Leave-One-Out Cross-Validation...")
        models = {"RandomForest": RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42),
                  "SVM (RBF)": SVC(kernel='rbf', random_state=42, class_weight='balanced')}
        loocv = LeaveOneOut()

        for feature_name, features in self.feature_sets.items():
            for model_name, model in models.items():
                print(f"  - Testing {model_name} on {feature_name} features...")
                cv_scores = cross_val_score(model, features, self.encoded_labels, cv=loocv, scoring='accuracy', n_jobs=-1)
                self.results[f"{model_name} - {feature_name}"] = {'cv_accuracy': np.mean(cv_scores)}
                print(f"    Mean CV Accuracy: {np.mean(cv_scores):.3f}")

    def generate_report(self):
        """Prints a summary of the model evaluation results."""
        print("\n--- Final Model Performance Report (LOOCV) ---")
        if not self.results: return

        results_data = [{'Model Configuration': k, 'Mean CV Accuracy': v['cv_accuracy']} for k,v in self.results.items()]
        results_df = pd.DataFrame(results_data).sort_values(by='Mean CV Accuracy', ascending=False).reset_index(drop=True)
        print(results_df.to_string(index=False, float_format='%.4f'))

        best = results_df.iloc[0]
        print(f"\nBest Performing Configuration:")
        print(f"  Model: {best['Model Configuration']}")
        print(f"  Accuracy Estimate: {best['Mean CV Accuracy']:.4f}")

    def analyze_class_distribution(self, stage="Initial"):
        """Prints a summary of the class distribution."""
        print(f"\n--- {stage} Class Distribution ---")
        class_counts = Counter(self.labels)
        print(f"Samples: {len(self.labels)}, Unique Classes: {len(class_counts)}")
        for cls, count in class_counts.most_common(5):
            print(f"  - Top class '{cls}': {count} samples")



# 6️⃣ Main Execution Pipeline

### 📖 Overview
The **end-to-end execution workflow** that orchestrates all components into a **unified multimodal machine learning pipeline**.  
This main execution sequence demonstrates the proper **integration of data loading, feature extraction, graph analysis, and model evaluation** for comprehensive **multimodal AI systems**.

---

## 🏗️ Pipeline Architecture

### 🚀 Execution Flow
**Pipeline Stages:**
1. **Environment Setup** → Google Drive mounting  
2. **Data Loading** → Raw images + JSON knowledge base  
3. **Feature Extraction** → Numerical representations  
4. **Graph Construction** → Similarity network building  
5. **Graph Analysis** → Structural feature extraction  
6. **Model Training** → Multimodal classifier evaluation  
7. **Results Reporting** → Performance analysis  

---

## Full End-to-End Pipeline


# Stage 1: Cloud Dataset Mount
```python
# Purpose: Establishes connection to cloud storage containing the dataset
# Critical: Force remount ensures fresh connection
# Environment: Google Colab notebook execution
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
```
# Stage 2: Data Loading and Validation
```python
from data_loader import EnhancedMultimodalDataLoader
loader = EnhancedMultimodalDataLoader(base_path='/content/drive/MyDrive/GNN_dataset/')
raw_dataset = loader.create_unified_dataset()
# Output: Dictionary with images, labels, Q&A pairs, knowledge base
```
# Stage 3: Feature Extraction Pipeline
```python
from feature_extractor import FeatureExtractor
extractor = FeatureExtractor(raw_dataset)
features_dict = extractor.create_all_features()
# Output: image_features, text_features, combined_features
```
# Stage 4: Graph Network Construction
```python
from graph_builder import GraphBuilder
graph_builder = GraphBuilder(features=features_dict['combined_features'], labels=features_dict['labels'])
graph_builder.build_similarity_graph(threshold=0.6)
graph_builder.visualize_graph('similarity', max_nodes=75)
# Output: Graph ready for feature extraction
```
# Stage 5: Graph Feature Integration
```python
from graph_analyzer import GraphAnalyzer
from trainer import RobustMultimodalTrainer
analyzer = GraphAnalyzer(graph_builder.graphs)
trainer = RobustMultimodalTrainer(features_dict=features_dict, graph_analyzer=analyzer)
# Output: Enhanced feature dictionary ready for training
```
# Stage 6: Label Simplification and Data Preparation
```python
trainer.merge_classes_by_prefix(separator='_')
trainer.prepare_data()
# Steps: Class merging, scaling, label encoding
```
# Stage 7: Model Evaluation and Reporting
```python
trainer.evaluate_all_models()
trainer.generate_report()
# Steps: LOOCV cross-validation, Random Forest & SVM evaluation
# Comparisons: Image-only, Text-only, Combined, Graph-Enhanced
# Output: Performance summary and best model identification

```

In [ ]:
if __name__ == '__main__':
    # Mount Google Drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)

    # STEP 1: Load raw data (images and JSON)
    loader = EnhancedMultimodalDataLoader(base_path='/content/drive/MyDrive/GNN_dataset/')
    raw_dataset = loader.create_unified_dataset()

    if raw_dataset:
        # STEP 2: Extract numerical features from the raw data
        extractor = FeatureExtractor(raw_dataset)
        features_dict = extractor.create_all_features()

        # STEP 3: Build and Visualize the Graph
        graph_builder = GraphBuilder(features=features_dict['combined_features'], labels=features_dict['labels'])
        graph_builder.build_similarity_graph(threshold=0.6)
        graph_builder.visualize_graph('similarity', max_nodes=75) # Display the graph
        analyzer = GraphAnalyzer(graph_builder.graphs)

        # STEP 4: Initialize the trainer with the numerical features and graph analyzer
        trainer = RobustMultimodalTrainer(features_dict=features_dict, graph_analyzer=analyzer)

        # STEP 5: Merge classes to simplify the problem (CRITICAL STEP)
        trainer.merge_classes_by_prefix(separator='_')

        # STEP 6: Prepare data for training and run the evaluation
        trainer.prepare_data()
        trainer.evaluate_all_models()

        # STEP 7: Generate the final report
        trainer.generate_report()
    else:
        print("\nCRITICAL ERROR: No data was loaded. The pipeline cannot continue.")

Mounting Google Drive...
Mounted at /content/drive

Verifying dataset paths...
Images path found: /content/drive/MyDrive/GNN_dataset/images
JSON path found: /content/drive/MyDrive/GNN_dataset/test

Loading images from subfolders...
  - Processing folder 'Apple,Alternaria Blotch': Found 2 images.
  - Processing folder 'Apple,Grey Spot': Found 2 images.
  - Processing folder 'Apple,Brown Spot': Found 2 images.
  - Processing folder 'Apple,Mosaic Virus': Found 2 images.
  - Processing folder 'Apple,Leaf Rust': Found 2 images.
  - Processing folder 'Apple,Cedar Apple Rust': Found 2 images.
  - Processing folder 'Apple,Powdery Mildew': Found 2 images.
  - Processing folder 'Apple,Healthy': Found 2 images.
  - Processing folder 'Apple,Frog Eye Leaf Spot': Found 2 images.
  - Processing folder 'Apple,Black Rot': Found 2 images.
  - Processing folder 'Bell Pepper,Bacterial Spot': Found 2 images.
  - Processing folder 'Apple,Scab': Found 2 images.
  - Processing folder 'Bell Pepper,Healthy': Fo

GraphAnalyzer initialized.

RobustMultimodalTrainer initialized.

Merging classes to simplify the classification task...

--- Before Merging Class Distribution ---
Samples: 114, Unique Classes: 60
  - Top class 'Apple,Alternaria Blotch': 2 samples
  - Top class 'Apple,Grey Spot': 2 samples
  - Top class 'Apple,Brown Spot': 2 samples
  - Top class 'Apple,Mosaic Virus': 2 samples
  - Top class 'Apple,Leaf Rust': 2 samples

--- After Merging Class Distribution ---
Samples: 114, Unique Classes: 60
  - Top class 'Apple,Alternaria Blotch': 2 samples
  - Top class 'Apple,Grey Spot': 2 samples
  - Top class 'Apple,Brown Spot': 2 samples
  - Top class 'Apple,Mosaic Virus': 2 samples
  - Top class 'Apple,Leaf Rust': 2 samples

Preparing and scaling all feature sets...
Extracting features from 'similarity' graph...
Graph-Enhanced features created, shape: (114, 1771)
Feature preparation complete.

Evaluating models with Leave-One-Out Cross-Validation...
  - Testing RandomForest on Image features..